# Predicting joint contact forces from gait inputs: OpenSim JCF data, NIH ACL cohort

Runtime > Change runtime type > Python 3.

## Setup

`numpy` for arrays, `pandas` to read the files, `matplotlib` to plot; all preinstalled on Colab. The colours are the Okabe-Ito palette, readable for colour-blind readers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPEED_COLOURS = {1.0: "#0072B2", 1.25: "#D55E00", 1.4: "#000000", 1.5: "#009E73"}  # one colour per belt speed
JOINT_COLOURS = {"hip": "#CC79A7", "knee": "#0072B2", "ankle": "#009E73"}

## Data

One folder, shared on Google Drive: four CSV files and `log.txt`, which defines every column. Read `log.txt` first.

One row is one stride of the right leg. `x-*.csv` holds the inputs (three ground reaction force curves, five joint angle curves, belt speed, mass, height), `y-*.csv` the outputs (the hip, knee and ankle joint contact force curves from OpenSim). Each curve is sampled at 101 points of the gait cycle, 0 to 100 %, heel strike to heel strike. Row i of x and row i of y are the same stride.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q

In [ ]:
print(open("data/log.txt").read())

In [ ]:
x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_test = pd.read_csv("data/x-test.csv")
y_test = pd.read_csv("data/y-test.csv")

## Explore

A data frame with one row per stride: three identifier columns, three scalar inputs, then the curves, one column per gait point.

In [ ]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
x_train.columns[:12].tolist()

In [ ]:
x_train.iloc[:5, :10]

In [ ]:
y_train.columns[:8].tolist()

In [ ]:
x_train["subject"].value_counts().sort_index()

In [ ]:
x_test["subject"].value_counts().sort_index()

In [ ]:
x_train["belt_speed_mps"].value_counts().sort_index()

In [ ]:
x_train.groupby("subject")[["mass_kg", "height_m"]].first()

## From wide columns to curves

A curve called `name` occupies the 101 columns `name_t000` to `name_t100`. `curve` returns it as a `(strides, 101)` array; `GRID` is 0, 1, ..., 100 percent of the gait cycle.

In [ ]:
GRID = np.arange(101)

INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]
OUTPUT_CURVES = ["hip_jcf_BW", "knee_jcf_BW", "ankle_jcf_BW"]

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

In [ ]:
grf_v = curve(x_train, "grf_vertical_BW")
grf_v.shape

In [ ]:
grf_v[0, [0, 15, 50, 62, 100]]  # one stride at heel strike, loading peak, push-off peak, toe-off, next heel strike

In [ ]:
knee = curve(y_train, "knee_jcf_BW")
knee.max(axis=1)[:10]  # peak knee JCF of the first ten strides, body weights

## Visualise the inputs

All 480 training strides of each input curve, coloured by belt speed, with the mean curve of each speed on top. Stance is roughly the first 62 % of the cycle (dotted line), swing the rest.

In [ ]:
def overlay(ax, df, name):
    C = curve(df, name)
    speeds = df["belt_speed_mps"].to_numpy()
    for v in sorted(SPEED_COLOURS):
        rows = speeds == v
        if rows.any():
            ax.plot(GRID, C[rows].T, color=SPEED_COLOURS[v], lw=0.5, alpha=0.25)
    for v in sorted(SPEED_COLOURS):
        rows = speeds == v
        if rows.any():
            ax.plot(GRID, C[rows].mean(axis=0), color=SPEED_COLOURS[v], lw=2.5, label=f"{v:g} m/s (n={rows.sum()})")
    ax.axvline(62, color="grey", lw=0.8, ls=":")
    ax.set_title(name, loc="left"); ax.set_xlim(0, 100); ax.grid(alpha=0.3)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, name in zip(axes.flat, INPUT_CURVES):
    overlay(ax, x_train, name)
axes[0][0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("Gait cycle [%]")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, name in zip(axes, ["grf_vertical_BW", "knee_flexion_deg", "ankle_angle_deg"]):
    overlay(ax, x_test, name)
    ax.set_xlabel("Gait cycle [%]")
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Test subjects", x=0.01, ha="left")
fig.tight_layout()

## Visualise the outputs

The joint contact force is the total force pressing the two bones of a joint together: the external load plus the compression produced by the muscles that cross the joint. That is why it reaches 3 to 6 body weights in walking while the ground reaction force is about 1 body weight. Two peaks in stance, near zero in swing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, name in zip(axes, OUTPUT_CURVES):
    overlay(ax, y_train, name)
    ax.set_xlabel("Gait cycle [%]")
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

In [ ]:
peaks = pd.DataFrame({name.split("_")[0]: curve(y_train, name).max(axis=1) for name in OUTPUT_CURVES})
peaks.describe().round(2)  # peak JCF per stride, body weights

## One stride, inputs and outputs together

The question the model has to answer: given the eight curves and the three scalars, produce the three JCF curves.

In [ ]:
i = 0  # row of the stride to show
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, name in zip(axes.flat, INPUT_CURVES):
    ax.plot(GRID, curve(x_train, name)[i], color="#0b0b0b", lw=2)
    ax.set_title(name, loc="left"); ax.set_xlim(0, 100); ax.grid(alpha=0.3)
fig.suptitle(f"{x_train.loc[i, 'subject']}, {x_train.loc[i, 'block']}, stride {x_train.loc[i, 'stride_in_block']}, {x_train.loc[i, 'belt_speed_mps']:g} m/s", x=0.01, ha="left")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name in OUTPUT_CURVES:
    j = name.split("_")[0]
    ax.plot(GRID, curve(y_train, name)[i], color=JOINT_COLOURS[j], lw=2.5, label=j)
ax.set_xlabel("Gait cycle [%]"); ax.set_ylabel("JCF [BW]"); ax.set_xlim(0, 100); ax.grid(alpha=0.3); ax.legend(frameon=False)
fig.tight_layout()

## Subject to subject

Knee JCF at 1.0 m/s, one panel per training subject, the mean of all training strides dashed. Each subject has a shape of their own; that between-subject variation is what makes prediction for a new person hard.

In [ ]:
subjects = sorted(x_train["subject"].unique())
fig, axes = plt.subplots(3, 4, figsize=(20, 11), sharex=True, sharey=True)
for ax, s in zip(axes.flat, subjects):
    rows = ((x_train["subject"] == s) & (x_train["belt_speed_mps"] == 1.0)).to_numpy()
    ax.plot(GRID, knee[rows].T, color=JOINT_COLOURS["knee"], lw=1, alpha=0.7)
    ax.plot(GRID, knee.mean(axis=0), color="grey", lw=1.5, ls="--")
    ax.set_title(s.replace("NIH_ACL_", ""), loc="left"); ax.grid(alpha=0.3)
for ax in axes[2]:
    ax.set_xlabel("Gait cycle [%]")
for ax in axes[:, 0]:
    ax.set_ylabel("Knee JCF [BW]")
fig.tight_layout()